#  Step 1: Install Dependencies
We will install the `ultralytics` package to get access to YOLO11, and `roboflow` to download your custom images later.

In [22]:
!pip install -q ultralytics roboflow
!pip install -q fiftyone

Now, let's import the necessary modules and verify that we have access to a GPU runtime, which will speed up our training significantly.

In [23]:
import torch
import ultralytics
#from ultralytics import YOLO

print(f"Ultralytics version: {ultralytics.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU: {torch.cuda.get_device_name(0)}")

Ultralytics version: 8.4.177
CUDA available: True
Active GPU: Tesla T4


# Step 2: Download Your Custom Roboflow Dataset
Use the snippet below to load your dataset.
Make sure to replace `YOUR_API_KEY`, `workspace-name`, `project-name`, and `version-number` with your actual Roboflow details.

In [4]:
from roboflow import Roboflow

# Initialize Roboflow client
rf = Roboflow(api_key="GOXcbY3plelrniCVW3q4")

# Leaving workspace() empty automatically selects your default workspace
project = rf.workspace().project("abdulghani")
version = project.version(3) # Change version number if needed

# Download dataset in YOLO format
dataset = version.download("yolov11")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to abdulghani-3 in yolov11:: 100%|██████████| 19836/19836 [00:02<00:00, 7825.52it/s] 


# Step 3: Configure Dataset and Merge Classes with COCO
Here we will define our target classes, and downlowd the classes from coco (COCO classes + your custom classes) and write a custom YAML configuration file for YOLO11 training.

In [27]:
%pip uninstall -y fiftyone

Found existing installation: fiftyone 1.22.1
Uninstalling fiftyone-1.22.1:
  Successfully uninstalled fiftyone-1.22.1


In [30]:
import sys
import importlib.metadata as md

print("Python:", sys.version)

for package in ["fiftyone", "Pillow", "ultralytics", "numpy"]:
    try:
        print(f"{package}: {md.version(package)}")
    except md.PackageNotFoundError:
        print(f"{package}: غير مثبت")

Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
fiftyone: غير مثبت
Pillow: 12.3.0
ultralytics: 8.4.177
numpy: 2.1.3


In [31]:
%pip install fiftyone==1.22.1

  Using cached fiftyone-1.22.1-py3-none-any.whl.metadata (23 kB)
Using cached fiftyone-1.22.1-py3-none-any.whl (24.0 MB)


In [6]:
import os, glob, shutil, yaml
import fiftyone as fo
import fiftyone.zoo as foz

# في Colab شغّل مرة واحدة في خلية مستقلة:  !pip install -q fiftyone

# ---------------- الإعدادات ----------------
dataset_dir = './abdulghani-1'                 # داتاسيت Roboflow
coco_classes = ["motorcycle", "car", "truck"]  # فئات COCO المطلوبة
N_TRAIN, N_VAL = 3000, 500                     # قلّلها/زدها بحسب حجم داتاسيتك

# الفئات النهائية (يجب أن تطابق ترقيم Roboflow)
class_names = {
    0: 'car',
    1: 'motorcycle',
    2: 'policemotorcycle',
    3: 'truck',
    4: 'van'
}

# ---------------- تحقق من ترقيم Roboflow ----------------
rf_yaml = os.path.join(dataset_dir, 'data.yaml')
if os.path.exists(rf_yaml):
    with open(rf_yaml) as f:
        rf_names = yaml.safe_load(f)['names']
    rf_names = dict(enumerate(rf_names)) if isinstance(rf_names, list) else rf_names
    if rf_names != class_names:
        print("تنبيه: ترقيم Roboflow يختلف عما كتبته، سأعتمد ترقيم Roboflow:")
        print(rf_names)
        class_names = rf_names
# توحيد أسماء الفئات لتجنب اختلاف الأحرف الكبيرة والصغيرة
name_to_id = {
    name.lower(): class_id
    for class_id, name in class_names.items()
}

# توحيد أسماء فئات COCO أيضًا
coco_classes_normalized = [
    name.lower() for name in coco_classes
]

# التحقق من وجود الفئات المطلوبة
missing = [
    name for name in coco_classes_normalized
    if name not in name_to_id
]

if missing:
    print("فئات COCO غير الموجودة في داتا سيت المشروع:", missing)
else:
    print("تم التحقق بنجاح: جميع فئات COCO المطلوبة موجودة.")

# ---------------- 1) تنزيل وتصدير من COCO ----------------
# (اسم split في FiftyOne: train / validation  ->  اسم التصدير: train / val)
export_root = "/content/coco_subset"
for zoo_split, out_split, n in [("train", "train", N_TRAIN),
                                ("validation", "val", N_VAL)]:
    ds = fo.zoo.load_zoo_dataset(
        "coco-2017",
        split=zoo_split,
        label_types=["detections"],
        classes=coco_classes,
        only_matching=True,       # يُبقي تسميات هذه الفئات فقط
        max_samples=n,
    )
    ds.export(
        export_dir=export_root,
        dataset_type=fo.types.YOLOv5Dataset,
        label_field="ground_truth",
        classes=coco_classes,     # ترقيم مؤقت: 0,1,2 بترتيب القائمة
        split=out_split,
    )

# ---------------- 2) الدمج مع إعادة الترقيم بالاسم ----------------
# ترقيم مؤقت (COCO) -> ترقيم نهائي (داتاسيتك)
remap = {i: name_to_id[c] for i, c in enumerate(coco_classes)}
print("إعادة الترقيم:", {coco_classes[i]: (i, "->", remap[i]) for i in remap})

def merge(coco_split, my_split):
    src_img = f"{export_root}/images/{coco_split}"
    src_lbl = f"{export_root}/labels/{coco_split}"
    dst_img = f"{dataset_dir}/{my_split}/images"
    dst_lbl = f"{dataset_dir}/{my_split}/labels"
    os.makedirs(dst_img, exist_ok=True)
    os.makedirs(dst_lbl, exist_ok=True)

    count = 0
    for img in glob.glob(f"{src_img}/*"):
        stem, ext = os.path.splitext(os.path.basename(img))
        lbl = f"{src_lbl}/{stem}.txt"
        if not os.path.exists(lbl):
            continue
        lines = []
        for line in open(lbl):
            p = line.split()
            if not p:
                continue
            p[0] = str(remap[int(p[0])])
            lines.append(" ".join(p))
        if not lines:
            continue
        shutil.copy(img, f"{dst_img}/coco_{stem}{ext}")
        with open(f"{dst_lbl}/coco_{stem}.txt", "w") as out:
            out.write("\n".join(lines) + "\n")
        count += 1
    print(f"{coco_split} -> {my_split}: أُضيفت {count} صورة")

merge("train", "train")
merge("val", "valid")

# حذف ملفات الكاش القديمة كي يعيد YOLO قراءة التسميات
for c in glob.glob(f"{dataset_dir}/**/*.cache", recursive=True):
    os.remove(c)

# ---------------- 3) ملف YAML ----------------
yaml_data = {
    'path': os.path.abspath(dataset_dir),
    'train': 'train/images',
    'val': 'valid/images',
    'test': 'test/images',
    'names': class_names
}
custom_yaml_path = 'custom_dataset.yaml'
with open(custom_yaml_path, 'w') as f:
    yaml.dump(yaml_data, f, default_flow_style=False)

print("تم حفظ:", os.path.abspath(custom_yaml_path))

تنبيه: ترقيم Roboflow يختلف عما كتبته، سأعتمد ترقيم Roboflow:
{0: 'Truck', 1: 'bus', 2: 'car', 3: 'motorcycle', 4: 'plicemotorsycle', 5: 'van'}
تم التحقق بنجاح: جميع فئات COCO المطلوبة موجودة.


INFO:fiftyone.zoo.datasets:Downloading split 'train' to '/root/fiftyone/coco-2017/train' if necessary


Found annotations at '/root/fiftyone/coco-2017/raw/instances_train2017.json'


INFO:fiftyone.utils.coco:Found annotations at '/root/fiftyone/coco-2017/raw/instances_train2017.json'


Sufficient images already downloaded


...

INFO:fiftyone.zoo.datasets:Dataset 'coco-2017-validation-500' created


Directory '/content/coco_subset' already exists; export will be merged with existing files


 100% |█████████████████| 500/500 [1.3s elapsed, 0s remaining, 427.8 samples/s]         


INFO:eta.core.utils: 100% |█████████████████| 500/500 [1.3s elapsed, 0s remaining, 427.8 samples/s]         


إعادة الترقيم: {'motorcycle': (0, '->', 3), 'car': (1, '->', 2), 'truck': (2, '->', 0)}
train -> train: أُضيفت 3000 صورة
val -> valid: أُضيفت 500 صورة
تم حفظ: /content/custom_dataset.yaml


# Step 4: Initialize and Train YOLO11 Model
Now, we initialize a pre-trained YOLO11 model and start training using our custom YAML file configuration.

In [7]:
import torch
import ultralytics
from ultralytics import YOLO

print(f"Ultralytics version: {ultralytics.__version__}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"Active GPU: {torch.cuda.get_device_name(0)}")

Ultralytics version: 8.4.177
PyTorch version: 2.11.0+cu130
CUDA available: True
Active GPU: Tesla T4


In [8]:
# Load a pre-trained YOLO11 nano detection model
model = YOLO('yolo11n.pt')

# Start training
results = model.train(
    data=custom_yaml_path,  # Path to our custom dataset config
    epochs=25,              # Number of epochs (adjust based on needs)
    imgsz=640,              # Image size
    batch=16,               # Batch size
    device=0 if torch.cuda.is_available() else 'cpu', # Train on GPU if available
    project='yolo11_custom_coco_training',
    name='run_1',

    freeze=10,                # تجميد أول 10 طبقات (الـ backbone)
    optimizer='AdamW',        # تحديد الـ optimizer صراحة، وإلا يُتجاهل lr0
    lr0=0.001,                # معدل تعلم أقل للحفاظ على ما تعلمه النموذج
    patience=15,              # إيقاف مبكر إذا لم تتحسن النتائج
)

Ultralytics 8.4.177 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=custom_dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=25, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=10, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=run_1, nbs=64, nms=None, opset=None, optimize=False, optimizer=AdamW, 